# Conversion Rate vs. Contact Metrics by Rental Location

This notebook analyzes the relationship between conversion rates and various contact metrics, grouped by rental location (RENT_LOC).

**Charts included:**
1. Conversion Rate vs. % Under 30 Minutes Contact
2. Conversion Rate vs. % Counter Contact
3. Conversion Rate vs. % No Contact
4. Conversion Rate vs. % Under 1 Hour Contact

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load data (INCLUDING UNUSED_IND=1 records)
file_path = '../data/raw/Conversion Data Nov-Dec 2025 (1).xlsx'
df = pd.read_excel(file_path, engine='openpyxl')

# Clean column names
df.columns = df.columns.str.strip()

# Note: Including ALL records (UNUSED_IND=0 and UNUSED_IND=1)
unused_count = df['UNUSED_IND'].sum() if 'UNUSED_IND' in df.columns else 0
print(f"Total records: {len(df):,}")
print(f"  - UNUSED_IND=1: {unused_count:,} ({unused_count/len(df)*100:.1f}%)")
print(f"  - UNUSED_IND=0: {len(df)-unused_count:,} ({(len(df)-unused_count)/len(df)*100:.1f}%)")
df.head()

In [ ]:
# Check contact time and group distributions
print("CONTACT RANGE Distribution:")
print(df['CONTACT RANGE'].value_counts())
print("\nCONTACT_GROUP Distribution:")
print(df['CONTACT_GROUP'].value_counts())

In [ ]:
# Calculate metrics by RENT_LOC
location_stats = df.groupby('RENT_LOC').agg(
    total_reservations=('RES_ID', 'sum'),
    conversions=('RENT_IND', 'sum'),
    under_30min_count=('CONTACT RANGE', lambda x: (x == '(a)<30min').sum()),
    under_1hr_count=('CONTACT RANGE', lambda x: (x.isin(['(a)<30min', '(b)31min - 1hr'])).sum()),
    no_contact_count=('CONTACT RANGE', lambda x: (x == 'NO CONTACT').sum()),
    counter_count=('CONTACT_GROUP', lambda x: (x == 'COUNTER').sum()),
    total_contacts=('CONTACT RANGE', 'count')
).reset_index()

# Calculate rates
location_stats['conversion_rate'] = (location_stats['conversions'] / location_stats['total_reservations'] * 100)
location_stats['pct_under_30min'] = (location_stats['under_30min_count'] / location_stats['total_contacts'] * 100)
location_stats['pct_under_1hr'] = (location_stats['under_1hr_count'] / location_stats['total_contacts'] * 100)
location_stats['pct_no_contact'] = (location_stats['no_contact_count'] / location_stats['total_contacts'] * 100)
location_stats['pct_counter'] = (location_stats['counter_count'] / location_stats['total_contacts'] * 100)

# Calculate majority GM for each location (handle NaN values)
def get_majority_value(x):
    mode_result = x.mode()
    if len(mode_result) > 0:
        return mode_result.iloc[0]
    return 'Unknown'

majority_gm = df.groupby('RENT_LOC')['GENERAL_MGR'].agg(get_majority_value).reset_index()
majority_gm.columns = ['RENT_LOC', 'majority_gm']
majority_gm['majority_gm'] = majority_gm['majority_gm'].fillna('Unknown')
location_stats = location_stats.merge(majority_gm, on='RENT_LOC', how='left')
location_stats['majority_gm'] = location_stats['majority_gm'].fillna('Unknown')

# Filter to locations with meaningful volume (at least 50 reservations)
location_stats_filtered = location_stats[location_stats['total_reservations'] >= 50].copy()

print(f"Total locations: {len(location_stats)}")
print(f"Locations with 50+ reservations: {len(location_stats_filtered)}")

# Get all unique GMs and create color mapping
all_gms = location_stats_filtered['majority_gm'].unique()
n_gms = len(all_gms)
print(f"\nUnique General Managers: {n_gms}")

# Generate distinct colors for all GMs using a combination of colormaps
import matplotlib.cm as cm
import numpy as np

# Use multiple colormaps to get enough distinct colors
colors_list = []
colormaps = ['tab20', 'tab20b', 'tab20c', 'Set1', 'Set2', 'Set3', 'Paired', 'Dark2']
for cmap_name in colormaps:
    cmap = cm.get_cmap(cmap_name)
    n_colors = cmap.N if hasattr(cmap, 'N') else 20
    for i in range(min(n_colors, 20)):
        colors_list.append(cmap(i / max(n_colors - 1, 1)))
    if len(colors_list) >= n_gms:
        break

# Build gm_colors dict for all GMs
gm_colors = {gm: colors_list[i % len(colors_list)] for i, gm in enumerate(sorted(all_gms))}

# Use majority_gm directly as the color group (no "Other" grouping)
location_stats_filtered['gm_color_group'] = location_stats_filtered['majority_gm']

print(f"\nMajority GM distribution (top 10):")
print(location_stats_filtered['majority_gm'].value_counts().head(10))

location_stats_filtered.head(10)

## Chart 1: Conversion Rate vs. % Under 30 Minutes Contact

In [ ]:
# Create scatter plot
fig, ax = plt.subplots(figsize=(14, 8))

# Size points by volume
sizes = location_stats_filtered['total_reservations'] / location_stats_filtered['total_reservations'].max() * 200 + 20

# Color points by majority GM
colors = location_stats_filtered['gm_color_group'].map(gm_colors)

scatter = ax.scatter(
    location_stats_filtered['pct_under_30min'],
    location_stats_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors.tolist(),
    edgecolors='black',
    linewidth=0.5
)

# Add legend for top GMs only (by location count)
top_gms_for_legend = location_stats_filtered['majority_gm'].value_counts().head(15).index.tolist()
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=gm_colors[gm], 
                              markersize=10, label=gm, markeredgecolor='black', markeredgewidth=0.5)
                  for gm in top_gms_for_legend]
ax.legend(handles=legend_handles, title='General Manager (Top 15)', loc='upper left', fontsize=7, bbox_to_anchor=(1.02, 1))

# Labels and title
ax.set_xlabel('% of Contacts Under 30 Minutes', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Quick Contact Rate by Rental Location\n(bubble size = volume, color = General Manager)', fontsize=14)

# Add grid
ax.grid(True, alpha=0.3)

# Add average lines
avg_conversion = location_stats_filtered['conversion_rate'].mean()
avg_under30 = location_stats_filtered['pct_under_30min'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conversion:.1f}%')
ax.axvline(x=avg_under30, color='blue', linestyle='--', alpha=0.5, label=f'Avg <30min: {avg_under30:.1f}%')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate correlations for all metrics
correlations = {
    '% Under 30min': location_stats_filtered['pct_under_30min'].corr(location_stats_filtered['conversion_rate']),
    '% Under 1hr': location_stats_filtered['pct_under_1hr'].corr(location_stats_filtered['conversion_rate']),
    '% Counter': location_stats_filtered['pct_counter'].corr(location_stats_filtered['conversion_rate']),
    '% No Contact': location_stats_filtered['pct_no_contact'].corr(location_stats_filtered['conversion_rate'])
}

print("Correlations with Conversion Rate:")
for metric, corr in correlations.items():
    print(f"  {metric}: {corr:.3f}")

## Chart 2: Conversion Rate vs. % Counter Contact

In [ ]:
# Chart 2: Conversion Rate vs. % Counter
fig, ax = plt.subplots(figsize=(14, 8))

sizes = location_stats_filtered['total_reservations'] / location_stats_filtered['total_reservations'].max() * 200 + 20

# Color points by majority GM
colors = location_stats_filtered['gm_color_group'].map(gm_colors)

scatter = ax.scatter(
    location_stats_filtered['pct_counter'],
    location_stats_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors.tolist(),
    edgecolors='black',
    linewidth=0.5
)

# Add legend for top GMs only
top_gms_for_legend = location_stats_filtered['majority_gm'].value_counts().head(15).index.tolist()
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=gm_colors[gm], 
                              markersize=10, label=gm, markeredgecolor='black', markeredgewidth=0.5)
                  for gm in top_gms_for_legend]
ax.legend(handles=legend_handles, title='General Manager (Top 15)', loc='upper left', fontsize=7, bbox_to_anchor=(1.02, 1))

ax.set_xlabel('% of Contacts via Counter', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Counter Contact Rate by Rental Location\n(bubble size = volume, color = General Manager)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_counter = location_stats_filtered['pct_counter'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conversion:.1f}%')
ax.axvline(x=avg_counter, color='blue', linestyle='--', alpha=0.5, label=f'Avg Counter: {avg_counter:.1f}%')

plt.tight_layout()
plt.show()

## Chart 3: Conversion Rate vs. % No Contact

In [ ]:
# Chart 3: Conversion Rate vs. % No Contact (0-20% range)
fig, ax = plt.subplots(figsize=(14, 8))

# Filter to locations with <=20% no contact
no_contact_filtered = location_stats_filtered[location_stats_filtered['pct_no_contact'] <= 20].copy()

sizes = no_contact_filtered['total_reservations'] / no_contact_filtered['total_reservations'].max() * 200 + 20

# Color points by majority GM
colors = no_contact_filtered['gm_color_group'].map(gm_colors)

scatter = ax.scatter(
    no_contact_filtered['pct_no_contact'],
    no_contact_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors.tolist(),
    edgecolors='black',
    linewidth=0.5
)

# Add legend for top GMs only
top_gms_for_legend = no_contact_filtered['majority_gm'].value_counts().head(15).index.tolist()
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=gm_colors[gm], 
                              markersize=10, label=gm, markeredgecolor='black', markeredgewidth=0.5)
                  for gm in top_gms_for_legend]
ax.legend(handles=legend_handles, title='General Manager (Top 15)', loc='upper left', fontsize=7, bbox_to_anchor=(1.02, 1))

ax.set_xlabel('% of No Contact', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. No Contact Rate by Rental Location\n(bubble size = volume, color = General Manager, 0-20% no contact range)', fontsize=14)

# Set x-axis limits
ax.set_xlim(0, 20)

ax.grid(True, alpha=0.3)

avg_no_contact = no_contact_filtered['pct_no_contact'].mean()
avg_conv_no_contact = no_contact_filtered['conversion_rate'].mean()
ax.axhline(y=avg_conv_no_contact, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conv_no_contact:.1f}%')
ax.axvline(x=avg_no_contact, color='blue', linestyle='--', alpha=0.5, label=f'Avg No Contact: {avg_no_contact:.1f}%')

# Show correlation for filtered data
corr = no_contact_filtered['pct_no_contact'].corr(no_contact_filtered['conversion_rate'])
ax.text(0.02, 0.98, f'Correlation: {corr:.3f}', transform=ax.transAxes, fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.show()

print(f"Showing {len(no_contact_filtered)} locations with <=20% no contact (excluded {len(location_stats_filtered) - len(no_contact_filtered)} locations)")

## Chart 4: Conversion Rate vs. % Under 1 Hour Contact

In [ ]:
# Chart 4: Conversion Rate vs. % Under 1 Hour
fig, ax = plt.subplots(figsize=(14, 8))

sizes = location_stats_filtered['total_reservations'] / location_stats_filtered['total_reservations'].max() * 200 + 20

# Color points by majority GM
colors = location_stats_filtered['gm_color_group'].map(gm_colors)

scatter = ax.scatter(
    location_stats_filtered['pct_under_1hr'],
    location_stats_filtered['conversion_rate'],
    s=sizes,
    alpha=0.6,
    c=colors.tolist(),
    edgecolors='black',
    linewidth=0.5
)

# Add legend for top GMs only
top_gms_for_legend = location_stats_filtered['majority_gm'].value_counts().head(15).index.tolist()
legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=gm_colors[gm], 
                              markersize=10, label=gm, markeredgecolor='black', markeredgewidth=0.5)
                  for gm in top_gms_for_legend]
ax.legend(handles=legend_handles, title='General Manager (Top 15)', loc='upper left', fontsize=7, bbox_to_anchor=(1.02, 1))

ax.set_xlabel('% of Contacts Under 1 Hour', fontsize=12)
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate vs. Under 1 Hour Contact Rate by Rental Location\n(bubble size = volume, color = General Manager)', fontsize=14)

ax.grid(True, alpha=0.3)

avg_under_1hr = location_stats_filtered['pct_under_1hr'].mean()
ax.axhline(y=avg_conversion, color='red', linestyle='--', alpha=0.5, label=f'Avg Conversion: {avg_conversion:.1f}%')
ax.axvline(x=avg_under_1hr, color='blue', linestyle='--', alpha=0.5, label=f'Avg <1hr: {avg_under_1hr:.1f}%')

plt.tight_layout()
plt.show()

In [ ]:
# Identify quadrants
location_stats_filtered['quadrant'] = 'Unknown'
location_stats_filtered.loc[
    (location_stats_filtered['conversion_rate'] >= avg_conversion) & 
    (location_stats_filtered['pct_under_30min'] >= avg_under30), 'quadrant'] = 'High Conv + Fast Contact'
location_stats_filtered.loc[
    (location_stats_filtered['conversion_rate'] >= avg_conversion) & 
    (location_stats_filtered['pct_under_30min'] < avg_under30), 'quadrant'] = 'High Conv + Slow Contact'
location_stats_filtered.loc[
    (location_stats_filtered['conversion_rate'] < avg_conversion) & 
    (location_stats_filtered['pct_under_30min'] >= avg_under30), 'quadrant'] = 'Low Conv + Fast Contact'
location_stats_filtered.loc[
    (location_stats_filtered['conversion_rate'] < avg_conversion) & 
    (location_stats_filtered['pct_under_30min'] < avg_under30), 'quadrant'] = 'Low Conv + Slow Contact'

print("Locations by Quadrant:")
location_stats_filtered['quadrant'].value_counts()

In [ ]:
# Top 10 locations by volume with all metrics
print("Top 10 Locations by Volume:")
location_stats_filtered.nlargest(10, 'total_reservations')[
    ['RENT_LOC', 'total_reservations', 'conversion_rate', 
     'pct_under_30min', 'pct_under_1hr', 'pct_counter', 'pct_no_contact']
].round(1)